# Tetrahedral MH and MH²M

Native volume and facet assembly with physical Neumann data. This notebook verifies an exact affine problem and displays a separately acquired nonaffine five-level campaign. The experiment extends the MH Robin formulation to three dimensions and uses the three-dimensional MH²M local maps; it is not a numerical table from either paper.

The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/19b93107ef55dd4704135aa4d1113f47e24e65250a3bddc5335f6ac2ec27abab/69_mh3d-companion.zip"
COMPANION_SHA256 = "19b93107ef55dd4704135aa4d1113f47e24e65250a3bddc5335f6ac2ec27abab"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/69_mh3d.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm.meshes.tetrahedron import TetraMesh
from pymhm.fem.traces.triangle_3d import TriangularSkeleton
from examples.formulations.robin import (
    define_robin_3d, robin_equations_3d, robin_gauge_3d, recover_robin_3d,
)


In [ ]:
from inspect import getsource
print(getsource(robin_equations_3d))
print(getsource(define_robin_3d))


For $K=I$, let $p=1+x+y+z$, $q=(-1,-1,-1)$ and $f=0$. Prescribe the outward physical flux on every boundary face and the exact volume mean $5/2$. The MH Robin multiplier differs from this flux by $p\sigma\cdot n$ and therefore needs degree one for this affine pressure, even though the physical normal flux is constant. The API applies that conversion through its boundary equations.

## Declare the three-field equations

The application declares the coupled local pressure/conormal saddle and its
global continuous pressure-trace test, using the independently chosen spaces:

$$
\begin{aligned}
A_T&=\begin{bmatrix}K_T&-\mathcal{B}_T\\-\mathcal{B}_T^{\mathsf T}&0\end{bmatrix},
&B_T&=\begin{bmatrix}0\\P_T\end{bmatrix},\\
C_T&=B_T^{\mathsf T}, &D_T&=0, &f_T&=\begin{bmatrix}F_T\\0\end{bmatrix}.
\end{aligned}
$$

Local coefficients are pressure and outward conormal eta=K grad(p).n; its
negative is physical Darcy normal flux. Gamma shares pressure nodes, while
Lambda remains private to each macrocell. The global `Equation` supplies
outward Neumann moments and fixed Gamma pressure values. Pure Neumann data add
the physical integral of the complete reconstructed pressure. Generic assembly
and solve consume these blocks directly. The shared Neumann-map kernel retains
the original boundary-mean basis for inspection and injectivity checks.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.three_field import define_three_field_3d, recover_three_field_3d

mesh = TetraMesh.unit_cube()
neumann = {int(face): -float(mesh.normals[face].sum()) for face in mesh.boundary_faces}
with threadpool_limits(1):
    robin_definition = define_robin_3d(
        mesh, neumann=neumann, mean_pressure=2.5, skeleton=TriangularSkeleton(mesh, degree=1)
    )
    robin_system = assemble(robin_definition.problem)
    mh = recover_robin_3d(
        robin_definition, robin_system,
        robin_system.solve(constraints=robin_gauge_3d(robin_definition, robin_system)),
    )
    mh2m_definition = define_three_field_3d(mesh, neumann=neumann, mean_pressure=2.5)
    mh2m_system = assemble(mh2m_definition.problem)
    mh2m = recover_three_field_3d(
        mh2m_definition,
        mh2m_system,
        mh2m_system.solve(constraints=pressure_constraints(mh2m_definition, mh2m_system)),
    )
    for method, result in (("MH", mh), ("MH2M", mh2m)):
        p_error = result.l2_error(lambda x: 1 + x.sum(axis=1), order=4)
        q_error = result.flux_l2_error([-1.0, -1.0, -1.0], order=4)
        balance = np.max(np.abs(result.conservation_residuals()))
        assert max(p_error, q_error, balance) < 2e-10
        print(method, {"pressure_L2": p_error, "flux_L2": q_error, "macro_balance": balance})


The archived refinement uses $p=1+x+2y+3z+\sin(\pi x)\sin(\pi y)\sin(\pi z)$, $K=I$, and nonzero Dirichlet data. MH uses local $P_3$ and $P_1$ Robin faces; MH²M uses local $P_2$, continuous $\Gamma=P_2$ and subdivided broken $\Lambda=P_1$. Errors are genuine volume integrals with independent orders 6 and 8. The two configurations have different local and global dimensions.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    record = json.loads((ROOT / "examples/results/mh3d/comparison.json").read_text())
    assert record["source_changed_during_run"] is False
    assert len(record["rows"]) == 10
    for row in record["rows"]:
        print(row["method"], row["resolution"], row["global_unknowns"], row["pressure_relative"], row["flux_relative"])
    display(Image(filename=str(ROOT / "docs/figures/mh3d/convergence.png")))

The section plots use one-sided fine-cell samples, with the actual macroface intersections in every panel. The signed $q_x$ component preserves direction; no averaging is applied across macrointerfaces.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    for name in ("mh-fields.png", "mh2m-fields.png"):
        display(Image(filename=str(ROOT / "docs/figures/mh3d" / name)))